# Video PPE Compliance Detection Pipeline

**Two-stage approach:**
1. **Stage 1** — Detect & track persons using the person detection model
2. **Stage 2** — For each detected person, crop and run PPE detection (helmet, jacket, boot)
3. **Annotate** the frame with compliance status per person

In [9]:
import cv2
import time
import numpy as np
from ultralytics import YOLO

In [10]:
# Load both models
person_detection_model = YOLO(model="/home/ai_vison/Desktop/PPE/model_finetune/person/runs/2026-06-04 16:54:10 150 l/weights/best.pt")
ppe_detection_model = YOLO(model="/home/ai_vison/Desktop/PPE/model_finetune/ppe_detect/runs/2026-06-02 18:26:19 200 l/weights/best.pt")

In [11]:
# PPE class mapping (from your PPE detection model)
PPE_CLASSES = {0: 'boot', 1: 'helmet', 2: 'jacket'}

# Required PPE items for full compliance
REQUIRED_PPE_FULL_PERSON = {'helmet', 'jacket', 'boot'}    # for class 1 (person - full body visible)
REQUIRED_PPE_HALF_PERSON = {'helmet', 'jacket'}             # for class 0 (half_person - upper body only)

# Person class mapping
PERSON_CLASSES = {0: 'half_person', 1: 'person'}

# Color coding
COLOR_COMPLIANT = (0, 200, 0)       # Green
COLOR_NON_COMPLIANT = (0, 0, 255)   # Red
COLOR_PARTIAL = (0, 165, 255)       # Orange

# PPE detection confidence threshold
PPE_CONF_THRESHOLD = 0.5

In [12]:
def detect_ppe_on_person(person_crop, person_cls):
    """
    Run PPE detection on a cropped person image.
    
    Args:
        person_crop: BGR image crop of a detected person
        person_cls: class id (0 = half_person, 1 = person)
    
    Returns:
        detected_ppe: set of detected PPE item names
        missing_ppe: set of missing PPE item names
        is_compliant: bool
        ppe_boxes: list of (x1, y1, x2, y2, class_name, conf) relative to the crop
    """
    # Run PPE model on the cropped person image
    ppe_results = ppe_detection_model.predict(
        source=person_crop,
        imgsz=[455, 171],   # same dimensions as your image pipeline
        classes=[0, 1, 2],  # boot, helmet, jacket
        conf=PPE_CONF_THRESHOLD,
        verbose=False
    )
    
    # Collect detected PPE items
    detected_ppe = set()
    ppe_boxes = []
    
    if ppe_results[0].boxes is not None and len(ppe_results[0].boxes) > 0:
        for box in ppe_results[0].boxes:
            cls_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            class_name = PPE_CLASSES.get(cls_id, 'unknown')
            detected_ppe.add(class_name)
            
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            ppe_boxes.append((x1, y1, x2, y2, class_name, conf))
    
    # Determine required PPE based on person type
    if person_cls == 1:  # full person
        required = REQUIRED_PPE_FULL_PERSON
    else:  # half person
        required = REQUIRED_PPE_HALF_PERSON
    
    missing_ppe = required - detected_ppe
    is_compliant = len(missing_ppe) == 0
    
    return detected_ppe, missing_ppe, is_compliant, ppe_boxes

In [13]:
# Open Video Stream (Replace with 0 for live webcam/robot camera)
video_path = "./inf_vid/Construction Safety Training clip 01.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video source.")
    exit()

# Get the video's original FPS and calculate the required delay
video_fps = cap.get(cv2.CAP_PROP_FPS)
frame_delay = int(1000 / video_fps) if video_fps > 0 else 33
print(f"Video FPS: {video_fps} | Calculated Delay: {frame_delay}ms")

Video FPS: 24.013261958211018 | Calculated Delay: 41ms


In [14]:
import os

output_dir = "./out_vid"
os.makedirs(output_dir, exist_ok=True)

input_name = os.path.basename(video_path)
output_path = os.path.join(output_dir, f"out_{input_name}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, video_fps, (frame_width, frame_height))

if not out.isOpened():
    raise RuntimeError(f"Could not open video writer for {output_path}")

print(f"Saving output video to: {output_path}")

Saving output video to: ./out_vid/out_Construction Safety Training clip 01.mp4


In [15]:
while cap.isOpened():
    start_time = time.time()

    ret, frame = cap.read()
    if not ret:
        print("Video stream finished.")
        break

    # ── Stage 1: Person Detection & Tracking ──
    results = person_detection_model.track(
        frame, persist=True, classes=[0, 1], conf=0.1, iou=0.7, verbose=False
    )

    if results[0].boxes is not None and len(results[0].boxes) > 0:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        class_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()

        track_ids = (
            results[0].boxes.id.cpu().numpy().astype(int)
            if results[0].boxes.id is not None
            else list(range(len(boxes)))
        )

        for box, track_id, person_cls, person_conf in zip(boxes, track_ids, class_ids, confs):
            x1, y1, x2, y2 = box
            # Clamp to frame boundaries
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)

            # Skip tiny crops that would cause issues
            if (x2 - x1) < 10 or (y2 - y1) < 10:
                continue

            # ── Stage 2: PPE Detection on cropped person ──
            person_crop = frame[y1:y2, x1:x2].copy()
            detected_ppe, missing_ppe, is_compliant, ppe_boxes = detect_ppe_on_person(
                person_crop, person_cls
            )

            # ── Annotation ──
            # Choose color based on compliance
            if is_compliant:
                box_color = COLOR_COMPLIANT
                status = "COMPLIANT"
            elif len(detected_ppe) > 0:
                box_color = COLOR_PARTIAL
                status = "PARTIAL"
            else:
                box_color = COLOR_NON_COMPLIANT
                status = "NON-COMPLIANT"

            # Draw person bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

            # Draw PPE item boxes on the main frame (offset to person position)
            for (px1, py1, px2, py2, ppe_name, ppe_conf) in ppe_boxes:
                abs_px1 = x1 + px1
                abs_py1 = y1 + py1
                abs_px2 = x1 + px2
                abs_py2 = y1 + py2
                cv2.rectangle(frame, (abs_px1, abs_py1), (abs_px2, abs_py2), (255, 255, 0), 1)
                cv2.putText(frame, f"{ppe_name} {ppe_conf:.1f}", (abs_px1, abs_py1 - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)

            # Build label text
            person_type = PERSON_CLASSES.get(person_cls, "person")
            ppe_list = ", ".join(sorted(detected_ppe)) if detected_ppe else "none"
            missing_list = ", ".join(sorted(missing_ppe)) if missing_ppe else ""

            label_line1 = f"ID:{track_id} [{person_type}] {status}"
            label_line2 = f"PPE: {ppe_list}"
            label_line3 = f"Missing: {missing_list}" if missing_list else ""

            # Draw background rectangle for text readability
            text_y = y1 - 10
            for line in [label_line1, label_line2, label_line3]:
                if not line:
                    continue
                (tw, th), _ = cv2.getTextSize(line, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(frame, (x1, text_y - th - 4), (x1 + tw + 4, text_y + 2), box_color, -1)
                cv2.putText(frame, line, (x1 + 2, text_y), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                text_y -= (th + 8)

    cv2.imshow("PPE Compliance Detection", frame)
    out.write(frame)

    # Dynamically calculate wait time to maintain natural playback speed
    elapsed_time = int((time.time() - start_time) * 1000)
    actual_delay = max(1, frame_delay - elapsed_time)

    if cv2.waitKey(actual_delay) & 0xFF == ord('q'):
        break

WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
WARNING ⚠️ imgsz=[455, 171] must be multiple of max st

In [16]:
# Clean up and release system resources
out.release()
cap.release()
cv2.destroyAllWindows()